# Svara (स्वर) — Multilingual, Voice-Personalised AAC Speech Assistant

Svara is a lightweight **Augmentative and Alternative Communication (AAC)** application designed to help non-verbal individuals (such as people with cerebral palsy, autism, post-stroke aphasia, or laryngectomy) communicate through natural Indian-language speech.

### Key Capabilities:
1. **Code-Mixed Speech:** Natural synthesis of bilingual Indian sentences (e.g. Roman and Devanagari Hinglish) in a unified voice without awkward language-switching artifacts.
2. **Voice Personalisation:** Allows users to choose an identity from Bodhan's catalog of Indic voices.
3. **Zero-Latency Playback via Caching:** Pre-caches high-frequency phrases locally for instantaneous response and offline availability.
4. **Explicit Tone/Style Control:** Lets users choose speaking styles (`happy`, `sad`, `anger`) to express emotional intent.

**Bodhan Model:** `indic-speak` (`POST /v1/audio/speech`)

**Requires:** `BODHAN_API_KEY` in `.env` (copy `.env.example`).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
from svara import SvaraClient

# Load credentials
load_dotenv()
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

client = SvaraClient()

## 1. Voice Personalisation

AAC is deeply personal — a child, an adult woman, and an elderly man should not have to share the exact same synthetic persona. `indic-speak` provides distinct male and female voices per language.

In [ ]:
phrase = "नमस्ते, मुझे आपकी मदद चाहिए।"
voices = ["Kavya", "Amit", "Suhani"]

for voice in voices:
    audio, is_cached, dt = client.synthesize(phrase, voice=voice, lang="hi", use_cache=False)
    p = OUT / f"voice_{voice}.wav"
    p.write_bytes(audio)
    print(f"{voice:10} -> {p.name} ({len(audio) // 1024} KB in {dt*1000:.0f} ms)")

## 2. Code-Mixed Speech (Hinglish)

Everyday Indian communication mixes native words and English freely. Svara supports both Roman transliteration and Devanagari script code-mixing rendered in a single coherent voice.

In [ ]:
codemix_samples = [
    ("roman_mix", "Mujhe thoda water chahiye, please."),
    ("native_mix", "मुझे थोड़ा water चाहिए, doctor ko call karo."),
    ("pure_hindi", "मुझे थोड़ा पानी चाहिए।"),
]

for label, text in codemix_samples:
    audio, is_cached, dt = client.synthesize(text, voice="Kavya", lang="hi", use_cache=False)
    p = OUT / f"{label}.wav"
    p.write_bytes(audio)
    print(f"{label:12} ({dt*1000:4.0f} ms): {text}")

## 3. Low-Latency AAC Caching Engine

In conversation, waiting 1.5–3 seconds for an API response creates conversational strain. Svara pre-caches frequently used board phrases locally. Tapping a cached phrase produces **0 ms** network latency and works offline.

In [ ]:
test_phrase = "Emergency hai, jaldi aao please."

# Call 1: Cold call (Live API request)
_, cached_1, latency_cold = client.synthesize(test_phrase, voice="Kavya", lang="hi", use_cache=True)
print(f"Cold Call (Live API):    {latency_cold*1000:.1f} ms (cached: {cached_1})")

# Call 2: Warm call (Local disk cache)
audio_cached, cached_2, latency_warm = client.synthesize(test_phrase, voice="Kavya", lang="hi", use_cache=True)
print(f"Warm Call (Local Cache): {latency_warm*1000:.1f} ms (cached: {cached_2})")

## 4. Emotional Expression with Speaking Styles

Svara allows the user to explicitly select an intended tone (`happy`, `sad`, `anger`) rather than having an AI guess or assume emotion.

In [ ]:
style_sentence = "ठीक है, मैं आ रहा हूँ।"
for style in ["neutral", "happy", "sad", "anger"]:
    audio, _, _ = client.synthesize(style_sentence, voice="Kavya", lang="hi", style=style)
    p = OUT / f"style_{style}.wav"
    p.write_bytes(audio)
    print(f"Generated style '{style}' -> {p.name}")

## 5. Listen to Samples

In [ ]:
from IPython.display import Audio, display

print("Code-mix sample:")
display(Audio(str(OUT / "roman_mix.wav")))

print("Personalised voice sample (Amit):")
display(Audio(str(OUT / "voice_Amit.wav")))

## 6. Next Steps & Interactive UI

To launch the full touch-friendly communication board, run the Streamlit app:

```bash
streamlit run app.py
```